# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [3]:
ПРІЗВИЩЕ_ІМЯ = 'Лопатін Володимир'
ГРУПА = 'ШІ-31'
НОМЕР = 12

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'


**Виконання: варіант 12.**

**Одиниці Д-димера враховано в етапі 1:** для Д-дим1 і Д-дим2 кожне
початкове число до 20 включно переводиться з мкг/мл у нг/мл множенням на 1000;
числа понад 20 уже вважаються нг/мл. Кілька вимірів усереднюються лише після
перерахунку кожного. Це правило надано студентом.
Початковий `df` показує оригінал Excel; для статистик і стандартизації
використовується очищена таблиця `дані`. Повне зіставлення Д-дим2 є
у виводі клітинки підготовки даних.

 Технічні розв’язки й пояснення заповнено.
Остання контрольна клітинка перевіряє також особисту декларацію.
Вона залишена без змін і до власного підтвердження студента повідомлятиме
про неготовність декларації. Її перевірки не обходяться.

In [4]:
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

pd.set_option('display.max_colwidth', 75)
with open('lr3_variants_student.json', encoding='utf-8-sig') as файл_варіантів:
    ВАРІАНТИ = json.load(файл_варіантів)
В = ВАРІАНТИ[НОМЕР - 1]
assert В['варіант'] == НОМЕР
print(f'ВАРІАНТ {НОМЕР} · {ПРІЗВИЩЕ_ІМЯ} · {ГРУПА}')
print('Ваші вісім стовпців:')
for i, назва in enumerate(В['стовпці'], 1):
    print(f'  {i}. {назва}')
print('Пара:', ' × '.join(В['пара']))
print('Індивідуальне питання:', В['питання'])


ВАРІАНТ 12 · Лопатін Володимир · ШІ-31
Ваші вісім стовпців:
  1. Телефон
  2. Примітка
  3. ступінь КТ
  4. Виписка
  5. spO2 випис
  6. АТ стац
  7. Д-дим2
  8. Гемогл2
Пара: ЛН × помер/жив
Індивідуальне питання: Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


In [5]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']
assert set(МОЇ + В['пара']).issubset(df.columns)
print('Розмір набору:', df.shape)
print('ПОЧАТКОВІ ДАНІ EXCEL: одиниці Д-димера ще не перераховано в цьому огляді. '
      'Результати перерахунку наведені в етапі 1.')
# Для показу маскуємо контакти; вихідний df залишається незмінним.
приклад = df[МОЇ].head(8).copy()
приклад['Телефон'] = приклад['Телефон'].map(
    lambda x: '<контакт приховано>' if pd.notna(x) else '<пропуск>'
)
display(приклад)


Розмір набору: (121, 77)
ПОЧАТКОВІ ДАНІ EXCEL: одиниці Д-димера ще не перераховано в цьому огляді. Результати перерахунку наведені в етапі 1.


,Телефон,Примітка,ступінь КТ,Виписка,spO2 випис,АТ стац,Д-дим2,Гемогл2
0,<контакт приховано>,NaN,сердній,2022-03-10 00:00:00,NaN,150/100,NaN,144
1,<контакт приховано>,"отримував актемра, часті носові кровотечі, втрата нюху",середній,2022-03-16 00:00:00,95,130/80,744,137
2,<контакт приховано>,часта втрата свідомосі в стаціонарі,критичний,2022-01-18 00:00:00,93,140/90,NaN,105
3,<контакт приховано>,панкреонекроз,критичний,2022-01-13 00:00:00,93,150/100,NaN,116
4,<контакт приховано>,NaN,NaN,2022-03-11 00:00:00,93,130/90,NaN,124
5,<контакт приховано>,NaN,середній,2021-12-26 00:00:00,NaN,170/100,NaN,120
6,<контакт приховано>,"глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",важкий,2022-05-11 00:00:00,96,130/80,NaN,151
7,<контакт приховано>,NaN,легкий,2022-05-28 00:00:00,94,120/80,1370,137


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [6]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Телефон     120 non-null    object
 1   Примітка    45 non-null     object
 2   ступінь КТ  42 non-null     object
 3   Виписка     120 non-null    object
 4   spO2 випис  68 non-null     object
 5   АТ стац     120 non-null    object
 6   Д-дим2      30 non-null     object
 7   Гемогл2     105 non-null    object
dtypes: object(8)
memory usage: 7.7+ KB


In [7]:
for c in МОЇ:
    з = df[c].dropna()
    типи = з.map(lambda x: type(x).__name__).value_counts()
    print(f'{c}: непорожніх {len(з)}/{len(df)}, унікальних {з.nunique()}')
    print('  Типи Python:', типи.to_dict())
    if c == 'Телефон':
        print('  Приклади контактів приховано; ідентифікатори не є вимірюваннями.')
    else:
        print('  Приклади:', [repr(v)[:85] for v in з.unique()[:4]])


Телефон: непорожніх 120/121, унікальних 120
  Типи Python: {'str': 97, 'int': 23}
  Приклади контактів приховано; ідентифікатори не є вимірюваннями.
Примітка: непорожніх 45/121, унікальних 44
  Типи Python: {'str': 45}
  Приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]
ступінь КТ: непорожніх 42/121, унікальних 12
  Типи Python: {'str': 42}
  Приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]
Виписка: непорожніх 120/121, унікальних 90
  Типи Python: {'datetime': 118, 'str': 2}
  Приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']
spO2 випис: непорожніх 68/121, унікальних 11
  Типи Python: {'int': 66, 'str': 2}
  Приклади: ['95', '93', '96', '94']
АТ стац: непорожніх 120/121, унікальних 21
  Типи Python: {'str': 120}

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [8]:
# Працюємо з окремою копією. df зберігає всі вихідні значення.
дані = df.copy(deep=True)
ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
ВІДПОВІДНІСТЬ_КТ = {
    'легкий': 'легкий', 'легка': 'легкий',
    'середній': 'середній', 'сердній': 'середній',
    'серед ступ': 'середній', 'серед ступеня': 'середній',
    'важкий': 'важкий', 'вкрай важкий': 'вкрай важкий',
    'критичний': 'критичний',
}
текст_кт = df['ступінь КТ'].astype('string').str.strip().str.lower()
невідомі_кт = set(текст_кт.dropna()) - set(ВІДПОВІДНІСТЬ_КТ)
assert not невідомі_кт, f'Потрібно перевірити назви ступенів: {невідомі_кт}'
дані['ступінь КТ'] = pd.Categorical(
    текст_кт.map(ВІДПОВІДНІСТЬ_КТ), categories=ПОРЯДОК_КТ, ordered=True
)

дані['Виписка'] = pd.to_datetime(
    df['Виписка'], format='mixed', dayfirst=True, errors='coerce'
)
хибні_дати = df['Виписка'].notna() & дані['Виписка'].isna()

# Беремо початкове число, зберігаючи окремо явну примітку про концентратор.
текст_spo2 = df['spO2 випис'].astype('string')
spo2_число = pd.to_numeric(
    текст_spo2.str.extract(r'^\s*(\d+(?:[.,]\d+)?)')[0]
              .str.replace(',', '.', regex=False), errors='coerce'
)
spo2_нуль = spo2_число.eq(0)
дані['spO2 випис'] = spo2_число.where(spo2_число.gt(0) & spo2_число.le(100))
# Булевий індикатор саме явної згадки, а не підтвердженого невикористання.
дані['with_concentrator'] = текст_spo2.str.contains(
    'на конц', case=False, na=False
).astype(bool)
явна_дотація = дані['with_concentrator']

def розділити_тиск(серія):
    компоненти = серія.astype('string').str.extract(
        r'^\s*(\d+(?:[.,]\d+)?)\s*/\s*(\d+(?:[.,]\d+)?)\s*$'
    )
    компоненти.columns = ['Систолічний', 'Діастолічний']
    return компоненти.apply(
        lambda с: pd.to_numeric(с.str.replace(',', '.', regex=False), errors='coerce')
    )

тиск_стац = розділити_тиск(df['АТ стац'])
дані['Гемогл2'] = pd.to_numeric(df['Гемогл2'], errors='coerce')
# Правило одиниць надав студент: до 20 включно — мкг/мл, понад 20 — нг/мл.
# Кожний компонент переводимо у нг/мл ДО усереднення значень однієї клітинки.
def компоненти_ддимера(значення):
    if pd.isna(значення):
        return []
    частини = [ч.strip() for ч in str(значення).split(';') if ч.strip()]
    assert частини and all(re.fullmatch(r'\d+(?:[.,]\d+)?', ч) for ч in частини), \
        f'Нерозпізнаний запис Д-димера: {значення!r}'
    return [float(ч.replace(',', '.')) for ч in частини]

журнал_ддимера = []
for стовпець in ['Д-дим1', 'Д-дим2']:
    результат = []
    for індекс, значення in df[стовпець].items():
        числа = компоненти_ддимера(значення)
        одиниці = ['мкг/мл' if ч <= 20 else 'нг/мл' for ч in числа]
        у_нг = [ч * 1000 if ч <= 20 else ч for ч in числа]
        середнє = float(np.mean(у_нг)) if у_нг else np.nan
        результат.append(середнє)
        if числа:
            журнал_ддимера.append({
                'Стовпець': стовпець, 'Рядок df': індекс,
                'Вихідний запис': значення, 'Компоненти': числа,
                'Одиниці за правилом студента': одиниці,
                'Компоненти, нг/мл': у_нг, 'Результат, нг/мл': середнє,
                'Кілька вимірів': len(числа) > 1,
                'Перераховано з мкг/мл': any(ч <= 20 for ч in числа),
            })
    дані[стовпець] = pd.Series(результат, index=df.index, dtype=float)
журнал_ддимера = pd.DataFrame(журнал_ддимера)
# Одиниці очищених стовпців зберігаємо явно разом із даними.
дані.attrs['одиниці'] = {'Д-дим1': 'нг/мл', 'Д-дим2': 'нг/мл'}


журнал = pd.DataFrame([
    ('ступінь КТ', int(df['ступінь КТ'].nunique()), int(дані['ступінь КТ'].nunique()),
     'Кількість різних записів до/після явного зіставлення назв'),
    ('Виписка', int(df['Виписка'].notna().sum()), int(дані['Виписка'].notna().sum()),
     'Дві нечитабельні дати залишено NaT; місяць не вгадуємо'),
    ('spO2 випис', int(df['spO2 випис'].notna().sum()), int(дані['spO2 випис'].notna().sum()),
     'Нулі відокремлено; явну згадку концентратора збережено'),
    ('АТ стац', int(df['АТ стац'].notna().sum()), int(тиск_стац.dropna().shape[0]),
     'Розділено на два числові компоненти'),
    ('Д-дим2', int(df['Д-дим2'].notna().sum()), int(дані['Д-дим2'].notna().sum()),
     'Усі значення у нг/мл за правилом студента; кілька вимірів усереднено'),
    ('Гемогл2', int(df['Гемогл2'].notna().sum()), int(дані['Гемогл2'].notna().sum()),
     'Текст «немає» залишено як пропуск'),
], columns=['Стовпець', 'До', 'Після', 'Пояснення'])
with pd.option_context('display.max_colwidth', None):
    display(журнал)
print('Нечитабельні дати:', df.loc[хибні_дати, 'Виписка'].tolist())
print('Нульових spO2 при виписці:', int(spo2_нуль.sum()))
print('Явних позначок концентратора:', int(дані['with_concentrator'].sum()))
print('with_concentrator: True — є явна згадка; False — згадки не знайдено, '
      'це не підтвердження відсутності концентратора.')
print('Записи з явною згадкою концентратора:')
display(pd.DataFrame({
    'Вихідний запис': df.loc[явна_дотація, 'spO2 випис'],
    'spO2 випис': дані.loc[явна_дотація, 'spO2 випис'],
    'with_concentrator': дані.loc[явна_дотація, 'with_concentrator'],
}))
print('Д-ДИМЕР: до 20 включно — мкг/мл × 1000; понад 20 — нг/мл. '
      'Усі результати нижче — нг/мл. Кілька вимірів усереднено ПІСЛЯ перерахунку.')
print('Усі 30 непорожніх записів Д-дим2: початковий запис → результат у нг/мл')
зіставлення_ддим2 = журнал_ддимера.loc[
    журнал_ддимера['Стовпець'].eq('Д-дим2'),
    ['Рядок df', 'Вихідний запис', 'Одиниці за правилом студента', 'Результат, нг/мл']
].reset_index(drop=True)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(зіставлення_ддим2)
print('Змінені записи обох стовпців:')
display(журнал_ддимера.loc[
    журнал_ддимера['Перераховано з мкг/мл'] | журнал_ддимера['Кілька вимірів']
])

,Стовпець,До,Після,Пояснення
0,ступінь КТ,12,5,Кількість різних записів до/після явного зіставлення назв
1,Виписка,120,118,Дві нечитабельні дати залишено NaT; місяць не вгадуємо
2,spO2 випис,68,58,Нулі відокремлено; явну згадку концентратора збережено
3,АТ стац,120,120,Розділено на два числові компоненти
4,Д-дим2,30,30,Усі значення у нг/мл за правилом студента; кілька вимірів усереднено
5,Гемогл2,105,104,Текст «немає» залишено як пропуск


Нечитабельні дати: ['24.024.2021', '20.24.2021']
Нульових spO2 при виписці: 10
Явних позначок концентратора: 2
with_concentrator: True — є явна згадка; False — згадки не знайдено, це не підтвердження відсутності концентратора.
Записи з явною згадкою концентратора:


,Вихідний запис,spO2 випис,with_concentrator
29,98 на конц,98,True
63,89 на конц,89,True


Д-ДИМЕР: до 20 включно — мкг/мл × 1000; понад 20 — нг/мл. Усі результати нижче — нг/мл. Кілька вимірів усереднено ПІСЛЯ перерахунку.
Усі 30 непорожніх записів Д-дим2: початковий запис → результат у нг/мл


,Рядок df,Вихідний запис,Одиниці за правилом студента,"Результат, нг/мл"
0,1,744,[нг/мл],744.00
1,7,1370,[нг/мл],1370.00
2,12,276,[нг/мл],276.00
3,13,0.9,[мкг/мл],900.00
4,37,987; 554,"[нг/мл, нг/мл]",770.50
5,39,986.93,[нг/мл],986.93
6,43,5829.55,[нг/мл],5829.55
7,44,0.25,[мкг/мл],250.00
8,45,2331.42,[нг/мл],2331.42
9,48,531,[нг/мл],531.00


Змінені записи обох стовпців:


,Стовпець,Рядок df,Вихідний запис,Компоненти,Одиниці за правилом студента,"Компоненти, нг/мл","Результат, нг/мл",Кілька вимірів,Перераховано з мкг/мл
12,Д-дим1,37,4442; 1084,"[4442.0, 1084.0]","[нг/мл, нг/мл]","[4442.0, 1084.0]",2763.0,True,False
50,Д-дим1,112,3.47,[3.47],[мкг/мл],[3470.0],3470.0,False,True
62,Д-дим2,13,0.9,[0.9],[мкг/мл],[900.0],900.0,False,True
63,Д-дим2,37,987; 554,"[987.0, 554.0]","[нг/мл, нг/мл]","[987.0, 554.0]",770.5,True,False
66,Д-дим2,44,0.25,[0.25],[мкг/мл],[250.0],250.0,False,True
71,Д-дим2,73,0.72,[0.72],[мкг/мл],[720.0],720.0,False,True


**Правила підготовки та межі інтерпретації.** Вихідні клітинки Excel не змінювалися.
Варіанти написання ступеня КТ зіставлено явно. Робочий порядок п’яти словесних
рівнів потрібен для навчального аналізу; відстані між ними не вважаються рівними.
Розрізнення «вкрай важкий» і «критичний» варто звірити зі словником джерела.

Дати `24.024.2021` і `20.24.2021` не виправлялися здогадками.
Нулі `spO2 випис` відокремлено від ненульових вимірів: усі десять нулів
належать записам з результатом «помер», тому вони можуть відображати результат,
а не порівнянний вимір сатурації. Це аналітичне правило, а не підтверджений код
пропуску. Сатурацію розділено на числовий стовпець `spO2 випис` і булевий
`with_concentrator`: True означає явну згадку «на конц» у вихідному записі,
False — відсутність такої згадки, зокрема в порожньому записі. False не доводить,
що концентратор не використовували. Обидві ознаки додано до навчальної матриці
ознак; їхня наявність не робить виміри в різних умовах автоматично порівнянними.

Для `Д-дим1` і `Д-дим2` застосовано надане студентом правило одиниць:
значення до 20 включно вважаються мкг/мл і множаться на 1000,
значення понад 20 — нг/мл. Усі робочі значення подано у нг/мл.
Це правило отримано від студента; в Excel окремих позначок одиниць немає.
Для кількох чисел через крапку з комою спочатку перераховано кожне,
потім обчислено середнє арифметичне за вказівкою студента.
Порожню частину після кінцевої крапки з комою не враховано.
Журнал зберігає початковий запис, компоненти, одиниці та результат.
Усереднення дає один показник на запис і не зберігає часової динаміки вимірів.
Статистики Д-димера інтерпретуються за умови правильності наданого правила.

In [9]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор',
        'Контактний запис ідентифікує людину; арифметичні різниці та відношення цифр не мають змісту.'),
    'Примітка': ('номінальна',
        'Вільний текст не має природного порядку; частоти точних записів не тотожні частотам захворювань.'),
    'ступінь КТ': ('порядкова',
        'Словесні рівні впорядковані за тяжкістю, але рівність відстаней між сусідніми рівнями не задана.'),
    'Виписка': ('інтервальна',
        'Для календарних дат змістовні різниці у днях; початок відліку умовний, тому відношення дат не має змісту.'),
    'spO2 випис': ('відношень',
        'Сатурація є часткою у відсотках із природним нулем; коди відсутності та умови виміру слід відокремити.'),
    'АТ стац': ('відношень',
        'Це складена ознака: два компоненти тиску є кількісними величинами; рядок виду 130/80 треба розділити.'),
    'Д-дим2': ('відношень',
        'Концентрація має природний нуль; значення приведено до нг/мл за правилом студента, кілька вимірів усереднено після перерахунку.'),
    'Гемогл2': ('відношень',
        'Концентрація гемоглобіну має природний нуль; запис «немає» є відсутністю виміру, а не нулем.'),
}
паспорт_df = pd.DataFrame(
    [(c, *ПАСПОРТ[c]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування']
)
with pd.option_context('display.max_colwidth', None):
    display(паспорт_df)


,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,Контактний запис ідентифікує людину; арифметичні різниці та відношення цифр не мають змісту.
1,Примітка,номінальна,Вільний текст не має природного порядку; частоти точних записів не тотожні частотам захворювань.
2,ступінь КТ,порядкова,"Словесні рівні впорядковані за тяжкістю, але рівність відстаней між сусідніми рівнями не задана."
3,Виписка,інтервальна,"Для календарних дат змістовні різниці у днях; початок відліку умовний, тому відношення дат не має змісту."
4,spO2 випис,відношень,Сатурація є часткою у відсотках із природним нулем; коди відсутності та умови виміру слід відокремити.
5,АТ стац,відношень,Це складена ознака: два компоненти тиску є кількісними величинами; рядок виду 130/80 треба розділити.
6,Д-дим2,відношень,"Концентрація має природний нуль; значення приведено до нг/мл за правилом студента, кілька вимірів усереднено після перерахунку."
7,Гемогл2,відношень,"Концентрація гемоглобіну має природний нуль; запис «немає» є відсутністю виміру, а не нулем."


In [10]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [11]:
ЧАСТОТИ = {}

def числовий_підсумок(з, cv=False):
    з = з.dropna().astype(float)
    if з.empty:
        return {'непорожніх': 0}
    р = {
        'непорожніх': len(з), 'мінімум': з.min(), 'максимум': з.max(),
        'медіана': з.median(), 'Q1': з.quantile(0.25), 'Q3': з.quantile(0.75),
        'середнє': з.mean(), 'стандартне відхилення (ddof=1)': з.std(ddof=1)
    }
    if cv:
        р['CV, %'] = (100 * з.std(ddof=1) / з.mean()
                      if len(з) > 1 and з.mean() != 0 else np.nan)
    return р

def допустима_статистика(стовпець, шкала):
    """Обчислює лише показники, що мають зміст для заданої ознаки."""
    з = дані[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()
    частоти = з.value_counts()
    ЧАСТОТИ[стовпець] = частоти

    if частоти.empty:
        return р
    моди = частоти[частоти.eq(частоти.max())].index.tolist()
    if частоти.max() == 1:
        р['мода'] = 'Усі непорожні значення унікальні; виділеної моди немає'
    elif ш == 'ідентифікатор':
        р['мода'] = f'Є повторення ідентифікаторів: максимальна частота {частоти.max()}'
    else:
        р['мода'] = моди
    р['частоти'] = f'Повна таблиця: ЧАСТОТИ[{стовпець!r}]'

    if ш in ('номінальна', 'ідентифікатор'):
        return р
    if ш == 'порядкова':
        впорядковані = з.sort_values().reset_index(drop=True)
        # Квантиль — перший рівень, що покриває задану частку спостережень.
        for назва, q in [('Q1', 0.25), ('медіана', 0.5), ('Q3', 0.75)]:
            позиція = max(0, int(np.ceil(q * len(впорядковані))) - 1)
            р[назва] = впорядковані.iloc[позиція]
        return р
    if стовпець == 'Виписка':
        р.update({
            'мінімальна дата': з.min(), 'максимальна дата': з.max(),
            'середня дата': з.mean(), 'медіанна дата': з.median(),
            'Q1': з.quantile(0.25), 'Q3': з.quantile(0.75),
            'стандартне відхилення, дні':
                ((з - з.min()).dt.total_seconds() / 86400).std(ddof=1),
        })
        return р  # CV календарних дат не обчислюємо.
    if стовпець == 'АТ стац':
        return {
            'Систолічний': числовий_підсумок(тиск_стац['Систолічний'], cv=True),
            'Діастолічний': числовий_підсумок(тиск_стац['Діастолічний'], cv=True),
        }
    if стовпець == 'Д-дим2':
        р['одиниця'] = 'нг/мл'
        р['умова інтерпретації'] = 'Правило студента: до 20 включно — мкг/мл; понад 20 — нг/мл; кілька вимірів усереднено.'
    р.update(числовий_підсумок(з, cv=ш in ('відношень', 'абсолютна')))
    return р

СТАТИСТИКИ = {}
for c in МОЇ:
    СТАТИСТИКИ[c] = допустима_статистика(c, ПАСПОРТ[c][0])
    одиниця = f", {дані.attrs['одиниці'][c]}" if c in дані.attrs['одиниці'] else ''
    print(f'--- {c}{одиниця} ({ПАСПОРТ[c][0]})')
    print(СТАТИСТИКИ[c])
    частоти = ЧАСТОТИ[c]
    if c == 'Телефон':
        print('Розподіл частот повторення контактів:', частоти.value_counts().to_dict())
    else:
        display(частоти.head(5).rename('Частота').to_frame())
    print()


--- Телефон (ідентифікатор)
{'непорожніх': 120, 'унікальних': 120, 'мода': 'Усі непорожні значення унікальні; виділеної моди немає', 'частоти': "Повна таблиця: ЧАСТОТИ['Телефон']"}
Розподіл частот повторення контактів: {1: 120}

--- Примітка (номінальна)
{'непорожніх': 45, 'унікальних': 44, 'мода': ['Поліалергетик'], 'частоти': "Повна таблиця: ЧАСТОТИ['Примітка']"}


,Частота
Примітка,
Поліалергетик,2
часта втрата свідомосі в стаціонарі,1
ЦД вперше виявлений,1
"2-Інфаркти, 2 стенти",1
"2-Інфаркти, 1 стент. ЦД вперше виявлений",1



--- ступінь КТ (порядкова)
{'непорожніх': 42, 'унікальних': 5, 'мода': ['середній'], 'частоти': "Повна таблиця: ЧАСТОТИ['ступінь КТ']", 'Q1': 'середній', 'медіана': 'середній', 'Q3': 'важкий'}


,Частота
ступінь КТ,
середній,18
важкий,10
легкий,7
вкрай важкий,4
критичний,3



--- Виписка (інтервальна)
{'непорожніх': 118, 'унікальних': 88, 'мода': [Timestamp('2021-09-07 00:00:00'), Timestamp('2022-06-06 00:00:00'), Timestamp('2022-04-06 00:00:00'), Timestamp('2021-11-19 00:00:00'), Timestamp('2021-09-11 00:00:00'), Timestamp('2021-09-15 00:00:00')], 'частоти': "Повна таблиця: ЧАСТОТИ['Виписка']", 'мінімальна дата': Timestamp('2021-04-27 00:00:00'), 'максимальна дата': Timestamp('2022-06-22 00:00:00'), 'середня дата': Timestamp('2022-01-29 18:06:06.101694976'), 'медіанна дата': Timestamp('2022-03-15 00:00:00'), 'Q1': Timestamp('2021-11-16 18:00:00'), 'Q3': Timestamp('2022-04-27 18:00:00'), 'стандартне відхилення, дні': 109.52398452859602}


...

,Частота
АТ стац,
130/80,63
120/80,12
140/90,10
150/100,6
140/80,6



--- Д-дим2, нг/мл (відношень)
{'непорожніх': 30, 'унікальних': 30, 'мода': 'Усі непорожні значення унікальні; виділеної моди немає', 'частоти': "Повна таблиця: ЧАСТОТИ['Д-дим2']", 'одиниця': 'нг/мл', 'умова інтерпретації': 'Правило студента: до 20 включно — мкг/мл; понад 20 — нг/мл; кілька вимірів усереднено.', 'мінімум': 210.0, 'максимум': 5829.55, 'медіана': 1038.5, 'Q1': np.float64(726.0), 'Q3': np.float64(2492.355), 'середнє': np.float64(1814.6113333333335), 'стандартне відхилення (ddof=1)': 1655.9078965302585, 'CV, %': np.float64(91.25413613991124)}


,Частота
Д-дим2,
744.00,1
1370.00,1
3378.00,1
392.00,1
377.76,1



--- Гемогл2 (відношень)
{'непорожніх': 104, 'унікальних': 59, 'мода': [144.0, 142.0], 'частоти': "Повна таблиця: ЧАСТОТИ['Гемогл2']", 'мінімум': 51.0, 'максимум': 190.0, 'медіана': 134.0, 'Q1': np.float64(116.0), 'Q3': np.float64(144.0), 'середнє': np.float64(128.71153846153845), 'стандартне відхилення (ddof=1)': 23.514372005521224, 'CV, %': np.float64(18.269047426970022)}


,Частота
Гемогл2,
144.0,5
142.0,5
137.0,4
116.0,4
136.0,4


---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [12]:
print('Опис ПОЧАТКОВОГО df до очищення та узгодження одиниць. '
      'Статистики очищеного Д-дим2 в нг/мл наведені в етапі 2.')
df[МОЇ].describe(include='all').T

Опис ПОЧАТКОВОГО df до очищення та узгодження одиниць. Статистики очищеного Д-дим2 в нг/мл наведені в етапі 2.


,count,unique,top,freq
Телефон,120,120,0958469788 дружина Ганна,1
Примітка,45,44,Поліалергетик,2
ступінь КТ,42,12,середній,10
Виписка,120,90,2021-09-07 00:00:00,3
spO2 випис,68,11,97,12
АТ стац,120,21,130/80,63
Д-дим2,30,30,744,1
Гемогл2,105,60,144,5


In [13]:
# Навмисно некоректні показники — лише для демонстрації помилки.
телефон_як_число = pd.to_numeric(df['Телефон'], errors='coerce').dropna()
кт_коди = дані['ступінь КТ'].cat.codes.replace(-1, np.nan)
дати_виписки = дані['Виписка'].dropna()
дні_від_1970 = (дати_виписки - pd.Timestamp('1970-01-01')).dt.total_seconds() / 86400
дні_від_2021 = (дати_виписки - pd.Timestamp('2021-01-01')).dt.total_seconds() / 86400
cv_дат_1970 = 100 * дні_від_1970.std(ddof=1) / дні_від_1970.mean()
cv_дат_2021 = 100 * дні_від_2021.std(ddof=1) / дні_від_2021.mean()

БЕЗГЛУЗДІ = [
    ('Телефон', 'Середнє числових контактних записів', телефон_як_число.mean(),
     f'Цифри є ідентифікатором, а не виміром; число залежить від запису контактів. '
     f'До того ж конверсія залишила лише {len(телефон_як_число)} із 120 непорожніх записів.'),
    ('ступінь КТ', 'Середнє довільних кодів 0–4', кт_коди.mean(),
     'Ранги зберігають порядок, але не відстані; середнє кодів не є середнім ступенем КТ.'),
    ('Виписка', 'CV календарних дат від епохи 1970, %', cv_дат_1970,
     f'Нуль календаря умовний: ті самі дати від епохи 2021 дають CV={cv_дат_2021:.3f}%. '
     'Висновок про однорідність не може залежати від вибору початку календаря.'),
]
assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(БЕЗГЛУЗДІ,
        columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде']))


,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,Середнє числових контактних записів,7.587990e+08,"Цифри є ідентифікатором, а не виміром; число залежить від запису контактів. До того ж конверсія залишила лише 23 із 120 непорожніх записів."
1,ступінь КТ,Середнє довільних кодів 0–4,1.476190e+00,"Ранги зберігають порядок, але не відстані; середнє кодів не є середнім ступенем КТ."
2,Виписка,"CV календарних дат від епохи 1970, %",5.757828e-01,Нуль календаря умовний: ті самі дати від епохи 2021 дають CV=27.815%. Висновок про однорідність не може залежати від вибору початку календаря.


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [14]:
A, B = В['пара']
ПОРЯДОК_ЛН = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']
ШКАЛИ_ПАРИ = {A: 'порядкова', B: 'номінальна, бінарна'}
print('Ваша пара:', A, '×', B)
print('Шкали:', ШКАЛИ_ПАРИ)
assert set(df[A].dropna()).issubset(ПОРЯДОК_ЛН)
assert set(df[B].dropna()).issubset({1, 2})
print('Кодування результату за поясненням у наданому звіті: 1 — помер, 2 — вижив.')
display(df[[A, B]].head(6))


Ваша пара: ЛН × помер/жив
Шкали: {'ЛН': 'порядкова', 'помер/жив': 'номінальна, бінарна'}
Кодування результату за поясненням у наданому звіті: 1 — помер, 2 — вижив.


,ЛН,помер/жив
0,ІІІ-IV,1
1,ІІ-ІІІ,2
2,ІІ,2
3,ІІІ-IV,2
4,ІІ-ІІІ,2
5,ІІ-ІІІ,1


Пара `ЛН × помер/жив` не входить до восьми призначених стовпців,
тому її шкали визначено окремо. V Крамера описує силу асоціації,
не використовуючи припущень про відстані між ступенями ЛН.

In [15]:
ОБРАНА_МІРА = 'Коефіцієнт Крамера V та критерій χ² незалежності'
ЧОМУ_САМЕ_ВОНА = (
    'ЛН має порядкову шкалу, а результат — бінарну номінальну. '
    'За таблицею методички використовуємо V Крамера: він оцінює асоціацію '
    'без припущення рівних відстаней між ступенями ЛН. Порядок при цьому не використовується.'
)
assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

пара = df[[A, B]].dropna()
спряженість = pd.crosstab(пара[A], пара[B]).reindex(ПОРЯДОК_ЛН)
chi2, p_асимптотичне, dof, очікувані = stats.chi2_contingency(
    спряженість, correction=False
)
n_пари = int(спряженість.to_numpy().sum())
V = np.sqrt(chi2 / (n_пари * min(спряженість.shape[0]-1, спряженість.shape[1]-1)))
display(спряженість.rename(columns={1: 'Помер', 2: 'Вижив'}))
print(f'n={n_пари}; χ²={chi2:.6f}; df={dof}; V={V:.6f}')
print(f'Асимптотичне p={p_асимптотичне:.3g}')
print(f'Очікуваних частот <5: {(очікувані < 5).sum()} із {очікувані.size}')

# За малих частот додатково оцінюємо p на випадкових таблицях
# із такими самими сумами рядків і стовпців.
кількість_симуляцій = 19999
випадкові_таблиці = stats.random_table.rvs(
    спряженість.sum(axis=1).to_numpy(),
    спряженість.sum(axis=0).to_numpy(),
    size=кількість_симуляцій, random_state=np.random.default_rng(12)
)
chi2_випадкові = (((випадкові_таблиці - очікувані) ** 2) / очікувані).sum(axis=(1, 2))
екстремальних = int(np.count_nonzero(chi2_випадкові >= chi2 - 1e-12))
p_MC = (екстремальних + 1) / (кількість_симуляцій + 1)
print(f'Монте-Карло: екстремальних таблиць {екстремальних}; p_MC={p_MC:.6f}')
print('Мінімальний крок оцінки p:', 1 / (кількість_симуляцій + 1))
print('У вибірці спостерігається виразна асоціація; причинність із цього не випливає.')


помер/жив,Помер,Вижив
ЛН,,
І,0,12
І-ІІ,0,9
ІІ,0,21
ІІ-ІІІ,49,10
ІІІ,11,6
ІІІ-IV,1,2


n=121; χ²=69.580071; df=5; V=0.758315
Асимптотичне p=1.25e-13
Очікуваних частот <5: 4 із 12
Монте-Карло: екстремальних таблиць 0; p_MC=0.000050
Мінімальний крок оцінки p: 5e-05
У вибірці спостерігається виразна асоціація; причинність із цього не випливає.


**Малі частоти.** У таблиці є очікувані частоти нижче 5.
Асимптотичне p наведено з цим обмеженням; додатковий розрахунок Монте-Карло
зберігає суми рядків і стовпців. Його роздільність — 1/20000;
нуль екстремальних симуляцій не означає справжнє p=0.
Джерела: [χ² та малі частоти](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.chi2_contingency.html),
[V Крамера](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.contingency.association.html).

In [16]:
# Два кодування зберігають один і той самий порядок ЛН.
рівні_коди = dict(zip(ПОРЯДОК_ЛН, [1, 2, 3, 4, 5, 6]))
нерівні_коди = dict(zip(ПОРЯДОК_ЛН, [1, 2, 3, 4, 5, 100]))
r_рівні = пара[A].map(рівні_коди).corr(пара[B])
r_нерівні = пара[A].map(нерівні_коди).corr(пара[B])
display(pd.DataFrame({
    'Кодування ЛН': ['1, 2, 3, 4, 5, 6', '1, 2, 3, 4, 5, 100'],
    'Пірсон r': [r_рівні, r_нерівні]
}))
print('Порядок категорій і самі спостереження не змінилися, а Пірсон змінився.')
print('Рівність інтервалів між ступенями не задана: це не інваріантна міра для цієї шкали.')


,Кодування ЛН,Пірсон r
0,"1, 2, 3, 4, 5, 6",-0.544224
1,"1, 2, 3, 4, 5, 100",0.009833


Порядок категорій і самі спостереження не змінилися, а Пірсон змінився.
Рівність інтервалів між ступенями не задана: це не інваріантна міра для цієї шкали.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [17]:
ПІДГОТОВКА = {
    'Телефон': 'Виключити з матриці ознак: це ідентифікатор і можливе джерело запам’ятовування.',
    'Примітка': 'Після перевірки часу запису — текстові ознаки або індикатор наявності; не кодувати довільні тексти порядковими числами.',
    'ступінь КТ': 'Зіставити написання, задати впорядковану категорію; коди зберігають порядок, але не доводять рівності інтервалів.',
    'Виписка': 'Перетворити на дату, відокремити помилки; обчислити тривалість від поступлення без CV календарних дат.',
    'spO2 випис': 'Розділити на числову сатурацію та булевий with_concentrator (явна згадка); відокремити нулі. False не доводить відсутності концентратора; враховувати умови виміру.',
    'АТ стац': 'Розділити на систолічний і діастолічний компоненти; перевірити формат та варіативність кожного.',
    'Д-дим2': 'Привести до нг/мл за правилом студента (до 20 включно — мкг/мл); кілька вимірів усереднити після перерахунку; перевірити варіативність і стандартизувати. Зберегти журнал.',
    'Гемогл2': 'Текст «немає» замінити пропуском у копії; перевірити варіативність і стандартизувати числові виміри.',
}
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
        columns=['Стовпець', 'Шкала', 'Як готуємо']))


,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,Виключити з матриці ознак: це ідентифікатор і можливе джерело запам’ятовування.
1,Примітка,номінальна,Після перевірки часу запису — текстові ознаки або індикатор наявності; не кодувати довільні тексти порядковими числами.
2,ступінь КТ,порядкова,"Зіставити написання, задати впорядковану категорію; коди зберігають порядок, але не доводять рівності інтервалів."
3,Виписка,інтервальна,"Перетворити на дату, відокремити помилки; обчислити тривалість від поступлення без CV календарних дат."
4,spO2 випис,відношень,Розділити на числову сатурацію та булевий with_concentrator (явна згадка); відокремити нулі. False не доводить відсутності концентратора; враховувати умови виміру.
5,АТ стац,відношень,Розділити на систолічний і діастолічний компоненти; перевірити формат та варіативність кожного.
6,Д-дим2,відношень,Привести до нг/мл за правилом студента (до 20 включно — мкг/мл); кілька вимірів усереднити після перерахунку; перевірити варіативність і стандартизувати. Зберегти журнал.
7,Гемогл2,відношень,Текст «немає» замінити пропуском у копії; перевірити варіативність і стандартизувати числові виміри.


Перевірка варіативності потрібна **до** ділення на std або max−min.
Для навчальної демонстрації нижче параметри обчислено на наявній вибірці.
У прогностичному застосуванні їх навчають на train і застосовують до test
без повторного підлаштування. Джерело:
[запобігання витоку даних](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [18]:
# Навчальна демонстрація, а не навчання прогностичної моделі.
# Параметри перетворень у реальній моделі оцінюють тільки на train.
ознаки = pd.DataFrame(index=df.index)

# Номінальна ознака: лише факт наявності тексту, без клінічної інтерпретації.
наявність_тексту = df['Примітка'].notna().map({True: 'є запис', False: 'немає запису'})
ознаки = ознаки.join(pd.get_dummies(наявність_тексту, prefix='Примітка', dtype=int))

# Порядкова шкала: пропуски не перетворюємо на окремий нижчий ранг.
ознаки['КТ_ранг'] = дані['ступінь КТ'].cat.codes.replace(-1, pd.NA).astype('Int64')

# Інтервальна шкала дат: різниця дат дає тривалість.
тривалість = (дані['Виписка'] - df['Поступлення']).dt.total_seconds() / 86400
відємні_тривалості = тривалість.lt(0)
ознаки['Тривалість_дні'] = тривалість.where(~відємні_тривалості)
print('Від’ємних тривалостей залишено на перевірку джерела:', int(відємні_тривалості.sum()))

# Шкала відношень: перед стандартизацією перевіряємо ненульове std.
гемоглобін = дані['Гемогл2']
середнє_hb = гемоглобін.mean()
std_hb = гемоглобін.std(ddof=0)
assert гемоглобін.notna().sum() > 1 and std_hb > 0
ознаки['Гемогл2_z'] = (гемоглобін - середнє_hb) / std_hb

ознаки['АТ_сист'] = тиск_стац['Систолічний']
ознаки['АТ_діаст'] = тиск_стац['Діастолічний']

# Число та умови виміру зберігаємо окремо; False означає лише відсутність згадки.
ознаки['spO2 випис'] = дані['spO2 випис']
ознаки['with_concentrator'] = дані['with_concentrator']

# Д-димер після узгодження одиниць за правилом студента.
ддимер = дані['Д-дим2']
std_dd = ддимер.std(ddof=0)
assert ддимер.notna().sum() > 1 and std_dd > 0
ознаки['Д-дим2_z'] = (ддимер - ддимер.mean()) / std_dd
print('Реалізовано підготовку номінальної, порядкової, інтервальної та шкали відношень.')
print('Телефон у матрицю не включено.')
display(ознаки.head(8))
print('Це перетворення для пояснення шкал; виписка та повторні аналізи '
      'недоступні для прогнозу на момент госпіталізації.')


Від’ємних тривалостей залишено на перевірку джерела: 2
Реалізовано підготовку номінальної, порядкової, інтервальної та шкали відношень.
Телефон у матрицю не включено.


,Примітка_немає запису,Примітка_є запис,КТ_ранг,Тривалість_дні,Гемогл2_z,АТ_сист,АТ_діаст,spO2 випис,with_concentrator,Д-дим2_z
0,1,0,1,14.0,0.653324,150,100,<NA>,False,NaN
1,0,1,1,22.0,0.354192,130,80,95,False,-0.657593
2,0,1,4,19.0,-1.013268,140,90,93,False,NaN
3,0,1,4,20.0,-0.543204,150,100,93,False,NaN
4,1,0,<NA>,27.0,-0.201339,130,90,93,False,NaN
5,1,0,1,11.0,-0.372271,170,100,<NA>,False,NaN
6,0,1,2,18.0,0.952456,130,80,96,False,NaN
7,1,0,0,10.0,0.354192,120,80,94,False,-0.273090


Це перетворення для пояснення шкал; виписка та повторні аналізи недоступні для прогнозу на момент госпіталізації.


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [19]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [20]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [21]:
ВИПРАВЛЕННЯ = [(1, 'Середнє кодів результату тлумачиться як проміжний стан.', 'Коди 1 і 2 позначають категорії. Їхнє середнє не означає стан «посередині», хоча за відомого кодування алгебраїчно пов’язане з часткою вижилих.', 'Навести частоти та частки результатів; частку померлих обчислювати індикатором результату 1.'), (2, 'Варіанти написання статі помилково оголошено різними статями.', 'Чотири різні рядки — це чотири текстові позначення. «ж» і «жін», а також «ч» і «чол» є скороченнями тих самих двох категорій.', 'Явно зіставити скорочення, перевірити залишкові назви й повторно обчислити частоти двох категорій.'), (3, 'Алфавітний порядок назв і середнє кодів КТ підміняють шкалу тяжкості.', 'sort_index сортує рядки за алфавітом, а довільні коди залежать від написання та синонімів. Їхня середня не є виміряним ступенем тяжкості.', 'Уніфікувати назви й задати природний порядок; показати частоти та порядкову медіану без середнього рангів.'), (4, 'to_numeric непомітно виключає текстові діапазони КТ.', 'Середнє обчислено лише за точковими числами, а інтервали та граничний запис втрачено. Це інша підвибірка; її число не характеризує весь стовпець.', 'Зберегти точкові значення та межі інтервалів, явно позначити неоднозначні одиниці й навести межі можливого середнього.'), (5, 'Сатурацію 0.9 змішано з відсотками, а середнє заміняє розподіл.', 'Один запис має інший порядок величини. Невідомо, чи це частка 90%, чи помилка. Саме середнє не доводить клінічний стан більшості пацієнтів.', 'Показати підозрілий запис і обидва сценарії — виключення та умовне перетворення на 90%; не ставити діагноз за середнім.'), (6, 'CV застосовано до календарної дати з умовним нулем.', 'Дата є інтервальною величиною. Зміна календарної епохи змінює середнє числових дат і CV, хоча часовий розкид залишається тим самим.', 'Навести мінімальну та максимальну дати, розмах і стандартне відхилення у днях; відмовитися від CV дат.'), (7, 'Малий коефіцієнт помилково використано як доказ відсутності зв’язку.', 'Для двох бінарних індикаторів Пірсон дорівнює φ, тому саме його обчислення допустиме. Помилка — необґрунтований висновок про відсутність асоціації.', 'Навести таблицю 2×2, φ або V та критерій незалежності. Не відхилити гіпотезу за p>0.05 не означає довести відсутність зв’язку.'), (8, 'Усі числові dtype оголошено успішно нормалізованими.', 'У ФЕРИТ2 тільки одне непорожнє значення, тому max−min=0 і нормалізація дає NaN. Числовий dtype також не гарантує придатність ознаки чи відсутність витоку.', 'Відокремити ціль, пропуски та сталі ознаки; перевірити діапазон перед діленням і оцінювати параметри масштабування на train.'), (9, 'Видалення всіх рядків із будь-яким пропуском названо очищенням.', 'dropna по 77 стовпцях залишає порожній набір. Відсутність NaN у нульовій кількості рядків не робить дані придатними для моделювання.', 'Вибрати ознаки під задачу, описати механізм пропусків, застосувати обґрунтовану обробку та перевірити кількість збережених рядків.'), (10, 'Дата розтину використана для прогнозування вже відомого результату.', 'Заповненість дати розтину повністю відображає факт смерті; перетворення NaT на мінімальний int64 додатково створює штучний числовий сигнал.', 'Виключити відомості після результату та визначити момент прогнозу; оцінювати модель лише за доступними на цей момент ознаками й належним розбиттям.')]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'
with pd.option_context('display.max_colwidth', None):
    display(pd.DataFrame(ВИПРАВЛЕННЯ,
        columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно']))


,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Середнє кодів результату тлумачиться як проміжний стан.,"Коди 1 і 2 позначають категорії. Їхнє середнє не означає стан «посередині», хоча за відомого кодування алгебраїчно пов’язане з часткою вижилих.",Навести частоти та частки результатів; частку померлих обчислювати індикатором результату 1.
1,2,Варіанти написання статі помилково оголошено різними статями.,"Чотири різні рядки — це чотири текстові позначення. «ж» і «жін», а також «ч» і «чол» є скороченнями тих самих двох категорій.","Явно зіставити скорочення, перевірити залишкові назви й повторно обчислити частоти двох категорій."
2,3,Алфавітний порядок назв і середнє кодів КТ підміняють шкалу тяжкості.,"sort_index сортує рядки за алфавітом, а довільні коди залежать від написання та синонімів. Їхня середня не є виміряним ступенем тяжкості.",Уніфікувати назви й задати природний порядок; показати частоти та порядкову медіану без середнього рангів.
3,4,to_numeric непомітно виключає текстові діапазони КТ.,"Середнє обчислено лише за точковими числами, а інтервали та граничний запис втрачено. Це інша підвибірка; її число не характеризує весь стовпець.","Зберегти точкові значення та межі інтервалів, явно позначити неоднозначні одиниці й навести межі можливого середнього."
4,5,"Сатурацію 0.9 змішано з відсотками, а середнє заміняє розподіл.","Один запис має інший порядок величини. Невідомо, чи це частка 90%, чи помилка. Саме середнє не доводить клінічний стан більшості пацієнтів.",Показати підозрілий запис і обидва сценарії — виключення та умовне перетворення на 90%; не ставити діагноз за середнім.
5,6,CV застосовано до календарної дати з умовним нулем.,"Дата є інтервальною величиною. Зміна календарної епохи змінює середнє числових дат і CV, хоча часовий розкид залишається тим самим.","Навести мінімальну та максимальну дати, розмах і стандартне відхилення у днях; відмовитися від CV дат."
6,7,Малий коефіцієнт помилково використано як доказ відсутності зв’язку.,"Для двох бінарних індикаторів Пірсон дорівнює φ, тому саме його обчислення допустиме. Помилка — необґрунтований висновок про відсутність асоціації.","Навести таблицю 2×2, φ або V та критерій незалежності. Не відхилити гіпотезу за p>0.05 не означає довести відсутність зв’язку."
7,8,Усі числові dtype оголошено успішно нормалізованими.,"У ФЕРИТ2 тільки одне непорожнє значення, тому max−min=0 і нормалізація дає NaN. Числовий dtype також не гарантує придатність ознаки чи відсутність витоку.","Відокремити ціль, пропуски та сталі ознаки; перевірити діапазон перед діленням і оцінювати параметри масштабування на train."
8,9,Видалення всіх рядків із будь-яким пропуском названо очищенням.,dropna по 77 стовпцях залишає порожній набір. Відсутність NaN у нульовій кількості рядків не робить дані придатними для моделювання.,"Вибрати ознаки під задачу, описати механізм пропусків, застосувати обґрунтовану обробку та перевірити кількість збережених рядків."
9,10,Дата розтину використана для прогнозування вже відомого результату.,Заповненість дати розтину повністю відображає факт смерті; перетворення NaT на мінімальний int64 додатково створює штучний числовий сигнал.,Виключити відомості після результату та визначити момент прогнозу; оцінювати модель лише за доступними на цей момент ознаками й належним розбиттям.


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

Нижче відтворено помилкові підходи з наданого аналізу та показано
коректні альтернативи для всіх десяти розділів — більше за потрібні три.
Статистичний аналіз навчальних записів не використовується для клінічного висновку.

In [22]:
# 1. Результат: частоти замість інтерпретації середнього номінальних кодів.
результат_частоти = df['помер/жив'].value_counts().sort_index()
print('1. Помилкова інтерпретація середнього кодів:', df['помер/жив'].mean())
display(pd.DataFrame({'Кількість': результат_частоти,
                      'Частка, %': 100 * результат_частоти / len(df)}))
print('Частка померлих:', df['помер/жив'].eq(1).mean())

# 2. Зіставлення скорочень статі.
стать_чиста = df['Стать'].str.strip().replace({'жін': 'ж', 'чол': 'ч'})
assert set(стать_чиста.dropna()) == {'ж', 'ч'}
print('2. Було текстових категорій:', df['Стать'].nunique())
print('Після зіставлення:', стать_чиста.value_counts().to_dict())

# 3. Алфавітні ранги та порядкова медіана — різні речі.
кт_алфавіт = sorted(df['ступінь КТ'].dropna().unique())
погані_коди_кт = df['ступінь КТ'].map(dict(zip(кт_алфавіт, range(len(кт_алфавіт)))))
print('3. Хибне середнє алфавітних кодів:', погані_коди_кт.mean())
print('Правильна порядкова медіана:', СТАТИСТИКИ['ступінь КТ']['медіана'])
display(дані['ступінь КТ'].value_counts(sort=False).rename('Частота').to_frame())

# 4. КТ: зберігаємо інтервали, а не замінюємо їх довільною точкою.
def межі_кт(значення):
    if pd.isna(значення):
        return (np.nan, np.nan)
    if isinstance(значення, (int, float, np.integer, np.floating)):
        return (float(значення), float(значення)) if 0 <= значення <= 1 else (np.nan, np.nan)
    текст = str(значення).strip().replace(',', '.')
    інтервал = re.fullmatch(r'(\d+(?:\.\d+)?)\s*[-–]\s*(\d+(?:\.\d+)?)\s*%', текст)
    if інтервал:
        return (float(інтервал[1])/100, float(інтервал[2])/100)
    верхня = re.fullmatch(r'до\s*(\d+(?:\.\d+)?)\s*%', текст)
    if верхня:
        return (0.0, float(верхня[1])/100)
    return (np.nan, np.nan)  # Наприклад, діапазон без одиниці.
кт_межі = pd.DataFrame(df['КТ'].map(межі_кт).tolist(),
                       columns=['Нижня', 'Верхня'], index=df.index)
кт_порівнювані = кт_межі.dropna()
погане_кт = pd.to_numeric(df['КТ'], errors='coerce')
print('4. Наївно збережено точкових записів:', погане_кт.notna().sum(),
      '; середнє, %:', 100 * погане_кт.mean())
print('Коректно подано точками або інтервалами:', len(кт_порівнювані))
print('Межі можливого середнього для цих записів, %:',
      (100 * кт_порівнювані.mean()).to_dict())
print('Неоднозначні записи без одиниці та порожні рядки потребують перевірки.')

# 5. Два прозорі сценарії для неоднозначної сатурації.
сатурація = df['spO2 без дот']
підозріла_сатурація = сатурація.gt(0) & сатурація.le(1)
print('5. Середнє як записано:', сатурація.mean())
print('Середнє без неоднозначного запису:', сатурація[~підозріла_сатурація].mean())
print('Сценарій «0.9 означає 90%», не підтверджене виправлення:',
      сатурація.where(~підозріла_сатурація, 100*сатурація).mean())

# 6. Часовий розкид у днях замість CV календарних дат.
дати_поступлення = pd.to_datetime(df['Поступлення'], errors='coerce').dropna()
зміщення_дні = (дати_поступлення - дати_поступлення.min()).dt.total_seconds() / 86400
print('6. Діапазон поступлень:', дати_поступлення.min(), '—', дати_поступлення.max())
print('Розмах, дні:', зміщення_дні.max(), '; std, дні:', зміщення_дні.std(ddof=1))

# 7. Для двох бінарних змінних r = φ. Висновок про відсутність не доведений.
таблиця_стать = pd.crosstab(стать_чиста, df['помер/жив'])
χ2_стать, p_стать, _, _ = stats.chi2_contingency(таблиця_стать, correction=False)
φ = стать_чиста.map({'ч': 0, 'ж': 1}).corr(df['помер/жив'].eq(1).astype(int))
V_стать = np.sqrt(χ2_стать / таблиця_стать.to_numpy().sum())
display(таблиця_стать)
print(f'7. φ={φ:.6f}; V={V_стать:.6f}; p={p_стать:.6f}')
print('За рівня 0.05 гіпотезу незалежності не відхилено; відсутність зв’язку не доведено.')

# 8. Порожні та сталі ознаки не масштабуємо.
числові_кандидати = df.select_dtypes('number').drop(columns=['помер/жив'])
кількість_значень = числові_кандидати.nunique()
сталі_ознаки = кількість_значень[кількість_значень.le(1)].index.tolist()
без_сталих = числові_кандидати.drop(columns=сталі_ознаки)
print('8. Виключені сталі або порожні ознаки:', сталі_ознаки)
print('Для ФЕРИТ2 непорожніх:', df['ФЕРИТ2'].notna().sum(),
      '; max-min:', df['ФЕРИТ2'].max()-df['ФЕРИТ2'].min())

# 9. Повне видалення по всій таблиці й по цільових полях дає різні вибірки.
print('9. Після dropna по всіх стовпцях:', len(df.dropna()))
для_ілюстрації = df[['Вік', 'Л1', 'помер/жив']].dropna()
print('Повні записи лише для Вік, Л1, результат:', len(для_ілюстрації))
print('Вибір ознак і стратегія пропусків мають відповідати конкретній задачі.')

# 10. Без використання мінімального int64 замість NaT показуємо сам витік.
дата_розтину_відома = df['Дата розтину'].notna().astype(int)
ціль_помер = df['помер/жив'].eq(1).astype(int)
print('10. Відома дата розтину = факт смерті для кожного рядка:',
      дата_розтину_відома.equals(ціль_помер))
print('Кореляція ознаки заповненості дати з результатом:',
      дата_розтину_відома.corr(ціль_помер))
доступні_на_вході = ['Вік', 'Стать', 'АТ пост', 'spO2 без дот']
print('Кандидати на вході до стаціонару, після перевірки часу виміру:',
      доступні_на_вході)
print('Виписка, дата розтину та наступні виміри для такого прогнозу виключаються.')


1. Помилкова інтерпретація середнього кодів: 1.4958677685950412


,Кількість,"Частка, %"
помер/жив,,
1,61,50.413223
2,60,49.586777


Частка померлих: 0.5041322314049587
2. Було текстових категорій: 4
Після зіставлення: {'ж': 75, 'ч': 46}
3. Хибне середнє алфавітних кодів: 5.904761904761905
Правильна порядкова медіана: середній


,Частота
ступінь КТ,
легкий,7
середній,18
важкий,10
вкрай важкий,4
критичний,3


4. Наївно збережено точкових записів: 27 ; середнє, %: 39.55555555555556
Коректно подано точками або інтервалами: 42
Межі можливого середнього для цих записів, %: {'Нижня': 40.1904761904762, 'Верхня': 43.33333333333334}
Неоднозначні записи без одиниці та порожні рядки потребують перевірки.
5. Середнє як записано: 82.04916666666666
Середнє без неоднозначного запису: 82.73109243697479
Сценарій «0.9 означає 90%», не підтверджене виправлення: 82.79166666666667
6. Діапазон поступлень: 2021-02-25 00:00:00 — 2022-06-03 00:00:00
Розмах, дні: 463.0 ; std, дні: 113.04812657257361


помер/жив,1,2
Стать,,
ж,43,32
ч,18,28


7. φ=0.176730; V=0.176730; p=0.051892
За рівня 0.05 гіпотезу незалежності не відхилено; відсутність зв’язку не доведено.
8. Виключені сталі або порожні ознаки: ['ФЕРИТ2']
Для ФЕРИТ2 непорожніх: 1 ; max-min: 0.0
9. Після dropna по всіх стовпцях: 0
Повні записи лише для Вік, Л1, результат: 117
Вибір ознак і стратегія пропусків мають відповідати конкретній задачі.
10. Відома дата розтину = факт смерті для кожного рядка: True
Кореляція ознаки заповненості дати з результатом: 1.0
Кандидати на вході до стаціонару, після перевірки часу виміру: ['Вік', 'Стать', 'АТ пост', 'spO2 без дот']
Виписка, дата розтину та наступні виміри для такого прогнозу виключаються.


---
# Етап 7. Висновки та індивідуальне питання

In [23]:
ВИСНОВКИ = [
    f'У варіанті 12 телефон є ідентифікатором, примітка — неупорядкованим текстом, '
    f'а ступінь КТ — порядковою ознакою: {df["ступінь КТ"].nunique()} написань '
    f'зіставлено з {дані["ступінь КТ"].nunique()} робочими рівнями без припущення рівних інтервалів.',
    f'Обробка виявила {int(хибні_дати.sum())} нечитабельні дати виписки, '
    f'{int(spo2_нуль.sum())} нульових записів spO2. Для обох стовпців Д-димера '
    f'застосовано правило студента: до 20 включно — мкг/мл, понад 20 — нг/мл; '
    f'кілька вимірів усереднено після приведення до нг/мл. '
    f'Для Д-дим2 збережено {дані["Д-дим2"].notna().sum()} записів; '
    f'середнє {СТАТИСТИКИ["Д-дим2"]["середнє"]:.2f} нг/мл за цим правилом.',
    f'Для ЛН і результату лікування отримано V Крамера {V:.4f} '
    f'при n={n_пари}. Через малі очікувані частоти додатково наведено p_MC={p_MC:.5f}; '
    f'асоціація в цій навчальній вибірці не доводить причинності.',
    f'Перекодування тих самих порядкових рівнів змінює Пірсона з {r_рівні:.4f} '
    f'на {r_нерівні:.4f}, хоча порядок не змінюється. Натомість повний dropna '
    f'по 77 ознаках залишає {len(df.dropna())} рядків, а дата розтину створює витік цілі.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), 'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')


1. У варіанті 12 телефон є ідентифікатором, примітка — неупорядкованим текстом, а ступінь КТ — порядковою ознакою: 12 написань зіставлено з 5 робочими рівнями без припущення рівних інтервалів.
2. Обробка виявила 2 нечитабельні дати виписки, 10 нульових записів spO2. Для обох стовпців Д-димера застосовано правило студента: до 20 включно — мкг/мл, понад 20 — нг/мл; кілька вимірів усереднено після приведення до нг/мл. Для Д-дим2 збережено 30 записів; середнє 1814.61 нг/мл за цим правилом.
3. Для ЛН і результату лікування отримано V Крамера 0.7583 при n=121. Через малі очікувані частоти додатково наведено p_MC=0.00005; асоціація в цій навчальній вибірці не доводить причинності.
4. Перекодування тих самих порядкових рівнів змінює Пірсона з -0.5442 на 0.0098, хоча порядок не змінюється. Натомість повний dropna по 77 ознаках залишає 0 рядків, а дата розтину створює витік цілі.


In [24]:
print('ВАШЕ ПИТАННЯ:', В['питання'])
тиск_пост = розділити_тиск(df['АТ пост'])
тиск_повні = тиск_пост.dropna()
assert (тиск_повні['Систолічний'] > тиск_повні['Діастолічний']).all()
середина_пари = тиск_пост.mean(axis=1, skipna=False)
пульсова_різниця = тиск_пост['Систолічний'] - тиск_пост['Діастолічний']
способи_АТ = pd.DataFrame({
    '1: систолічний': тиск_пост['Систолічний'],
    '1: діастолічний': тиск_пост['Діастолічний'],
    '2: пульсова різниця': пульсова_різниця,
    '3: арифметична середина пари': середина_пари,
    '3: пульсова різниця': пульсова_різниця,
})
display(способи_АТ.head(8))

ВІДПОВІДЬ = (
    'Рядок АТ пост спочатку розділяю за символом / і перевіряю обидва числа. '
    'Спосіб 1: дві ознаки — систолічний і діастолічний тиск. Це зберігає '
    'початкову інформацію й дозволяє окремо пояснити внесок кожного компонента. '
    'Спосіб 2: одна ознака S−D, пульсова різниця. Вона компактна, але втрачає '
    'загальний рівень: 140/100 і 120/80 мають однакову різницю 40. '
    'Спосіб 3: пара ((S+D)/2, S−D). Це оборотне математичне перекодування: '
    'S=середина+різниця/2, D=середина−різниця/2; середина є лише арифметичною ознакою. '
    'Для моделі обираю спосіб 1 як найбільш прозорий і без втрати інформації. '
    'Одночасно включати обидві повні пари в лінійну модель не слід через '
    'точну лінійну залежність. Пропуски залишаю явними; параметри масштабування '
    'за потреби оцінюю лише на навчальній частині після розбиття. '
    f'У файлі розпізнано {len(тиск_повні)} повних записів зі {len(df)}. '
    'Модель у цій лабораторній не навчається; перетворення показано на фактичних даних.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою'
print(ВІДПОВІДЬ)


ВАШЕ ПИТАННЯ: Стовпець «АТ пост» містить значення на кшталт «140/100». Запропонуйте три способи перетворити його на числові ознаки та обґрунтуйте, який із них візьмете в модель.


,1: систолічний,1: діастолічний,2: пульсова різниця,3: арифметична середина пари,3: пульсова різниця
0,140,100,40,120.0,40
1,130,80,50,105.0,50
2,150,100,50,125.0,50
3,170,110,60,140.0,60
4,160,100,60,130.0,60
5,140,90,50,115.0,50
6,130,80,50,105.0,50
7,140,80,60,110.0,60


Рядок АТ пост спочатку розділяю за символом / і перевіряю обидва числа. Спосіб 1: дві ознаки — систолічний і діастолічний тиск. Це зберігає початкову інформацію й дозволяє окремо пояснити внесок кожного компонента. Спосіб 2: одна ознака S−D, пульсова різниця. Вона компактна, але втрачає загальний рівень: 140/100 і 120/80 мають однакову різницю 40. Спосіб 3: пара ((S+D)/2, S−D). Це оборотне математичне перекодування: S=середина+різниця/2, D=середина−різниця/2; середина є лише арифметичною ознакою. Для моделі обираю спосіб 1 як найбільш прозорий і без втрати інформації. Одночасно включати обидві повні пари в лінійну модель не слід через точну лінійну залежність. Пропуски залишаю явними; параметри масштабування за потреби оцінюю лише на навчальній частині після розбиття. У файлі розпізнано 120 повних записів зі 121. Модель у цій лабораторній не навчається; перетворення показано на фактичних даних.


In [25]:
# Технічна перевірка не замінює особисту декларацію студента.
assert len(ПАСПОРТ) == 8 and set(ПАСПОРТ) == set(МОЇ)
assert len(ВИПРАВЛЕННЯ) == 10
assert n_пари == len(пара) == 121
assert np.isclose(V, stats.contingency.association(спряженість, method='cramer'))
assert 0 < p_MC <= 1
assert тиск_стац.notna().all(axis=1).sum() == df['АТ стац'].notna().sum()
assert 'Телефон' not in ознаки
# Перевіряємо перерахунок і всі показані підготовлені записи.
assert дані.attrs['одиниці'] == {'Д-дим1': 'нг/мл', 'Д-дим2': 'нг/мл'}
assert len(зіставлення_ддим2) == 30
assert np.allclose(зіставлення_ддим2['Результат, нг/мл'],
                   дані.loc[зіставлення_ддим2['Рядок df'], 'Д-дим2'])
assert np.isclose(дані.loc[13, 'Д-дим2'], 900)
assert np.isclose(дані.loc[73, 'Д-дим2'], 720)
assert np.isclose(дані.loc[39, 'Д-дим2'], 986.93)
assert np.isclose(СТАТИСТИКИ['Д-дим2']['середнє'], 1814.6113333333335)
assert np.isclose(дані.loc[44, 'Д-дим2'], 250)
assert np.isclose(дані.loc[112, 'Д-дим1'], 3470)
assert np.isclose(дані.loc[37, 'Д-дим2'], (987 + 554) / 2)
assert np.isclose(дані.loc[37, 'Д-дим1'], (4442 + 1084) / 2)
assert np.isclose(дані.loc[39, 'Д-дим1'], 480)
for стовпець in ['Д-дим1', 'Д-дим2']:
    assert дані[стовпець].notna().sum() == df[стовпець].notna().sum()
assert np.isclose(ознаки['Д-дим2_z'].dropna().mean(), 0, atol=1e-12)
assert np.isclose(ознаки['Д-дим2_z'].dropna().std(ddof=0), 1)
assert дані['with_concentrator'].dtype == bool
assert int(дані['with_concentrator'].sum()) == 2
pd.testing.assert_series_equal(
    дані['with_concentrator'],
    df['spO2 випис'].astype('string').str.contains(
        'на конц', case=False, na=False
    ).astype(bool).rename('with_concentrator')
)
pd.testing.assert_series_equal(ознаки['with_concentrator'], дані['with_concentrator'])
pd.testing.assert_series_equal(ознаки['spO2 випис'], дані['spO2 випис'])
assert np.isclose(ознаки['Гемогл2_z'].dropna().mean(), 0, atol=1e-12)
assert np.isclose(ознаки['Гемогл2_z'].dropna().std(ddof=0), 1)
pd.testing.assert_frame_equal(
    df, pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
)
print('Технічні перевірки пройдено. Вихідні дані не змінено.')

Технічні перевірки пройдено. Вихідні дані не змінено.


**Використання ШІ й особиста частина.** Код і текст підготовлено за допомогою
OpenAI Codex. Технічний запуск та перевірки виконує агент.
Студент після розбору сам обирає категорії декларації, описує власну перевірку
щонайменше 120 символами та надає особисте підтвердження.
Готові формулювання не підставлялися як свідчення його розуміння.

---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [28]:
ІНСТРУМЕНТ_ШІ = 'OpenAI Codex'
# Особисті поля заповнює студент після пояснення та власної перевірки.
# Категорія «код, перевірено» за умовою також означає, що студент розуміє код.
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних': 'код, перевірено',
    'етап 2 — допустимі статистики': 'код, перевірено',
    'етап 3 — безглузді числа': 'код, перевірено',
    'етап 4 — міра зв’язку': 'код, перевірено',
    'етап 5 — підготовка до моделі': 'код, перевірено',
    'етап 6 — виправлення аналізу': 'код, перевірено',
    'етап 7 — висновки й відповідь': 'код, перевірено',
}
ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'хід думок, щоб дійсно код виконував задачу цілком, коли агент стверджував якийсь факт, я гуглив і звіряв із даними з мережі'
ПІДТВЕРДЖУЮ = True


In [29]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 12 готова до захисту.
Лопатін Володимир, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.